# "THE PRICE IS RIGHT" 顶点项目

本周——基于抓取的 Amazon 数据，构建一个能根据描述预测某物价格的模型


一个能根据描述估算某物价格的模型。

# 日程安排

DAY 1：数据整理（Data Curation）  
DAY 2：数据预处理（Data Pre-processing）  
DAY 3：评估、基线、传统机器学习  
DAY 4：深度学习与 LLM  
DAY 5：微调前沿模型  

## DAY 4：神经网络与 LLM

今天我们将从传统机器学习走到神经网络，再到大语言模型！！

In [ ]:
# 导入

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
# 环境：加载 HF_TOKEN 并登录 Hugging Face

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
# 从 Hub 加载完整或精简定价数据集

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# 在看人工神经网络之前

## 还有另一种我们可以考虑的神经网络

In [ ]:
# 把测试集写入 CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# 再读回来

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
# 按测试集下标取人工标注价格

def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
# 对比人工预测与真实价格

human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


In [ ]:
# 在 100 条上评估「人类基线」

evaluate(human_pricer, test, size=100)

# 现在——一个普通的神经网络

在本课程余下部分，我们会更深入地了解神经网络如何工作，以及如何训练神经网络。

这只是一个预览——让我们用 PyTorch 从零搭建自己的神经网络。

用它建立直觉即可；此刻不必了解神经网络的全部细节……

In [ ]:
# 准备我们的文档与价格

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# 使用 HashingVectorizer 做词袋模型
# 对 CountVectorizer 使用 binary=True 会生成 “one-hot 向量”

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
# 定义神经网络——这里是用 PyTorch 创建 8 层神经网络的代码

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
# 将数据转换为 PyTorch 张量
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# 将数据拆分为训练集与验证集
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# 创建 loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 初始化模型
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
# 统计神经网络可训练参数量（模型容量的粗指标）

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

In [ ]:
# 定义损失函数与优化器

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 我们将完整遍历数据 2 次

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # 接下来 4 行是训练的 4 个阶段：前向传播、损失计算、反向传播、优化
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

In [ ]:
# 推理：summary → HashingVectorizer → 张量 → 网络输出；价格下限为 0

def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
# 评估神经网络报价器

evaluate(neural_network, test)

# 现在——走向前沿！

看看前沿模型开箱即用表现如何；不训练，只基于它们的世界知识做推理。

明天我们将做一些训练。

In [ ]:
# 构造给前沿 LLM 的消息：只让模型输出价格数字（英文提示勿改）

def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
# 看一条测试商品的 summary

print(test[0].summary)

In [ ]:
# 查看 messages 格式

messages_for(test[0])

In [ ]:
# gpt-4.1-nano 的函数

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
# 对单条测试样本调用 gpt-4.1-nano

gpt_4__1_nano(test[0])

In [ ]:
# 真实价格对照

test[0].price

In [ ]:
# 评估 gpt-4.1-nano（零样本 prompting，尚未微调）

evaluate(gpt_4__1_nano, test)

In [ ]:
# Claude Opus 报价函数

def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
# 评估 Claude

evaluate(claude_opus_4_5, test)

In [ ]:
# Gemini 3 Pro 报价（reasoning_effort 可调）

def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
# 评估 Gemini 3 Pro（样本与并发数限制，控制费用）

evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
# Gemini Flash Lite：更快更便宜的变体

def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
# 评估 Gemini Flash Lite

evaluate(gemini_2__5_flash_lite, test)

In [ ]:
# Grok 快速非推理模型报价

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
# 评估 Grok

evaluate(grok_4__1_fast, test)

In [ ]:
# gpt-5.1 的函数

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
# 评估 GPT-5.1

evaluate(gpt_5__1, test)